# 03 · Auditing the Relative Wealth Index (H1, H2, H3, H6)

Does Meta's Relative Wealth Index miss Indigenous communities? This notebook asks four questions: whether
Indigenous localities are on the map at all (H1); whether the map ranks Indigenous places as wealthier than
equally poor non-Indigenous places (H2); whether the definition of poverty changes who counts as poorest
(H3); and what the map's errors would cost under a realistic budget formula (H6).

The hypotheses, tests and model specifications were fixed in the pre-analysis plan
(`docs/pre_analysis_plan.md`) and tagged `pap-v1` before this analysis was run on real data; the tag's
date records that they came first. Any later change is listed in the plan's deviations log.

**Runtime:** about 5 minutes. **Outputs:** `results/real/tables/*.csv`, the figures, and the results page
`docs/results.md`.

In [ ]:
# Make the project importable; works in JupyterLab, VS Code and Colab without reinstalling.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from mapaudit import notebook as nb
from mapaudit import pipeline, report
from mapaudit.config import get_paths

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
MODE = nb.notebook_mode()  # "real" by default; CI sets MAPAUDIT_NOTEBOOK_MODE=demo (synthetic data)
print("Mode:", MODE)

In [ ]:
if MODE == "demo":
    nb.prepare_demo(through="build")
audit = pipeline.audit(MODE)
robust = pipeline.robustness(MODE)
report.report(MODE)
pd.Series(audit, name="value", dtype=object).to_frame()

## The two confirmatory tests

H1 is two-sided and H2 one-sided (coefficient > 0), both at α = 0.05 after Holm's correction, as specified
in the plan. Everything else in this notebook is an estimate with an uncertainty range, not a test.

In [ ]:
tests = nb.load_table("confirmatory_tests", MODE)
tests[["p_value", "p_holm"]] = tests[["p_value", "p_holm"]].map("{:.1e}".format)  # p-values can be tiny
tests

## Who the audit can see

Share of the rural population each part of the analysis covers. Localities with fewer than three dwellings
have suppressed census data, including Indigenous status, so they get their own row instead of silently
leaving the denominator.

In [ ]:
nb.load_table("evaluation_coverage", MODE).round(3)

## H1 · Who is on the map at all?

The share of rural localities and people inside a tile the map covers, by group and locality size, then a
linear probability model of coverage on Indigenous population share that holds locality size and
remoteness fixed. H1 is the coefficient on Indigenous share, tested two-sided with standard errors
clustered by municipality.

In [ ]:
nb.show_figure("rwi_coverage", MODE)
display(nb.load_table("coverage", MODE).round(3))
nb.load_table("coverage_regression", MODE).round(4)

## H2 · Is the rank right, within the same true wealth decile?

Rank error is the map's percentile minus the census percentile, so a value above zero means the map ranks
the place as wealthier than the census does. Controls are added one at a time (rurality, remoteness, true
wealth decile, state fixed effects) to show how much of the raw gap each one explains. The confirmatory
estimate is the last row of the regression table, with all controls.

In [ ]:
nb.show_figure("decile_errors", MODE)
nb.show_figure("audit_specs", MODE)
nb.load_table("audit_regressions", MODE).round(3)

## The user's view: among places the map ranks the same

The same errors, grouped by the map's decile instead of the census's. A gap here means a program that
targets by map rank treats unequally poor places as equals. Groups that differ in average wealth produce
some gap here even for a map with group-blind errors, so I read it against a no-bias benchmark computed
from these data: the map's values shuffled among tiles at the same census percentile, which keeps the
groups' real wealth difference and the map's real noise but removes any group-specific error. Only a gap
beyond the benchmark's range points to errors specific to Indigenous places.

In [ ]:
nb.show_figure("map_decile_errors", MODE)
display(nb.load_table("map_decile_errors", MODE).round(2))
keys = ["calibration_gap_pp", "calibration_gap_no_bias_pp", "calibration_gap_no_bias_range_pp"]
pd.Series({k: audit[k] for k in keys}, name="value", dtype=object).to_frame()

## Robustness (pre-specified)

The H2 estimate under each alternative promised in the plan: unweighted, tiles with at least 50 people,
Indigenous thresholds of 30% and 50%, the two other definitions of poverty (and the official CONAPO index
when it is available), and a language-based Indigenous share.

In [ ]:
robust.round(3)

## H3 · Does the definition of poverty change who counts as poorest?

The share of each group's population counted among the nation's poorest 20% under each definition of
poverty, and how closely the definitions agree (Spearman correlations), by group.

In [ ]:
nb.show_figure("label_choice", MODE)
display(nb.load_table("label_choice", MODE).round(3))
nb.load_table("label_agreement", MODE).round(3)

## H6 · What would the map's errors cost?

Two stylized rules, each applied once with the census and once with the map. **Targeting** funds the
localities predicted to be poorest until 20% of the population is covered; the exclusion rate is the share
of the truly poorest it leaves out. The **budget formula** splits a fixed budget across municipalities by
the estimated number of deprived people, a simplified version of Mexico's FAIS infrastructure fund. The
tables give the net change for Indigenous-majority municipalities and the misallocation share: the
fraction of the whole budget that would have to move to reproduce the census-based allocation.

In [ ]:
nb.show_figure("allocation", MODE)
display(nb.load_table("rwi_targeting", MODE).round(3))
display(nb.load_table("rwi_allocation", MODE).round(3))
nb.load_table("rwi_allocation_errors", MODE).round(3)

Where the map and the census disagree, region by region. Map tiles are averaged into hexagons about
40 km across (population-weighted), colored by Indigenous population share (left) and by rank error
(right). Hexagons holding fewer than five tiles are not drawn, so the map shows regional patterns and
no single community can be read off it.

In [ ]:
nb.show_figure("map", MODE)